# Intelligent Job Role Recommendation Using Artificial Neural Network (ANN)

**Multi-label resume-to-role recommendation | Python • Pandas • Scikit-learn • TensorFlow/Keras**

## 1. Project objective
Build an educational system that reads a candidate profile (skills, education, experience and resume text) and recommends several potentially suitable job roles. Unlike ordinary multiclass classification, one candidate can be suitable for several roles simultaneously.

> **Important dataset note:** The included CSV records are synthetic, programmatically generated examples. Their multi-label annotations are reproducible rule-based teaching labels, not human-verified hiring outcomes or a downloaded real-world resume corpus. The occupation catalog is a compact educational taxonomy. Do not describe these examples as real candidate data or as validated labour-market ground truth. Replace or augment them with properly licensed, consented and human-reviewed data before any real-world use.

## 2. Workflow
1. Load and inspect four CSV datasets.
2. Clean and normalize resume text and skill terms.
3. Join multi-label annotations to resume records.
4. Build numeric/categorical/text features with a `ColumnTransformer`.
5. Split data using a reproducible train/test split.
6. Train a Keras MLP with one sigmoid output per job role.
7. Evaluate with micro/macro precision, recall, F1, subset accuracy and per-role metrics.
8. Recommend ranked roles and explain skill gaps.
9. Try the customizable prediction snippets at the end.

## 3. Why multi-label classification?
A resume can fit Data Analyst, Business Intelligence Developer and Business Analyst at the same time. A softmax layer forces probabilities to compete and sum to one; this is not appropriate here. A sigmoid neuron per role produces an independent score in [0, 1] for each role. A threshold determines which roles are recommended.

## 4. Install dependencies
Run this once in a fresh environment. In Google Colab, TensorFlow and many data-science packages may already be installed.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn tensorflow

## 5. Imports and reproducibility

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, precision_score, recall_score,
                             f1_score, hamming_loss, accuracy_score)
from sklearn.metrics import multilabel_confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
DATA_DIR = "../data" if os.path.exists("../data/resume_profiles.csv") else "data"

## 6. Load CSV datasets

In [ ]:
resumes = pd.read_csv(os.path.join(DATA_DIR, "resume_profiles.csv"))
role_labels = pd.read_csv(os.path.join(DATA_DIR, "resume_role_labels.csv"))
occupations = pd.read_csv(os.path.join(DATA_DIR, "occupation_catalog.csv"))
skill_taxonomy = pd.read_csv(os.path.join(DATA_DIR, "skill_taxonomy.csv"))

print("Resume profiles:", resumes.shape)
print("Role annotations:", role_labels.shape)
print("Occupation catalog:", occupations.shape)
print("Skill taxonomy:", skill_taxonomy.shape)
display(resumes.head(3))
display(occupations.head())

## 7. Data quality checks and exploration

In [ ]:
print("Missing values in resume data:")
display(resumes.isna().sum().to_frame("missing_count"))
print("Duplicate resume IDs:", resumes["resume_id"].duplicated().sum())
print("Annotation label distribution:")
label_counts = role_labels.groupby("job_role")["is_relevant"].sum().sort_values(ascending=False)
display(label_counts.to_frame("positive_examples"))
print("Labels per resume:")
labels_per_resume = role_labels.groupby("resume_id")["is_relevant"].sum()
display(labels_per_resume.describe().to_frame("labels_per_resume"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
label_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Positive training annotations by job role")
ax.set_xlabel("Number of annotated resumes")
plt.tight_layout()
plt.show()

## 8. Clean text and normalize skill names

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9+#.\s-]", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def parse_skills(value):
    if pd.isna(value):
        return []
    return sorted({clean_text(s) for s in str(value).split(",") if clean_text(s)})

resumes["resume_text_clean"] = resumes["resume_text"].apply(clean_text)
resumes["skills_list"] = resumes["skills"].apply(parse_skills)
resumes["skills_normalized"] = resumes["skills_list"].apply(lambda x: " ".join(x))
resumes["education_level"] = resumes["education_level"].fillna("Unknown")
resumes["domain_interest"] = resumes["domain_interest"].fillna("Unknown")
resumes["years_experience"] = pd.to_numeric(resumes["years_experience"], errors="coerce")
display(resumes[["resume_id","education_level","years_experience","skills_normalized"]].head())

## 9. Construct the multi-label target matrix

In [ ]:
role_names = occupations["job_role"].tolist()
target_wide = (role_labels.pivot(index="resume_id", columns="job_role", values="is_relevant")
               .reindex(columns=role_names).fillna(0).astype("int32"))
model_df = resumes.set_index("resume_id").join(target_wide, how="inner").reset_index()
Y = model_df[role_names].to_numpy(dtype="float32")
print("Target matrix:", Y.shape)
print("Number of roles:", len(role_names))
print("Average labels per profile:", Y.sum(axis=1).mean().round(2))

## 10. Feature engineering and preprocessing
The pipeline combines TF-IDF text features, one-hot encoded education/domain, and a scaled numeric experience feature. The transformer is fitted only on training rows to avoid data leakage.

In [ ]:
text_col = "resume_text_clean"
categorical_cols = ["education_level", "domain_interest"]
numeric_cols = ["years_experience"]

preprocessor = ColumnTransformer(
    transformers=[
        ("resume_tfidf", TfidfVectorizer(max_features=2500, ngram_range=(1,2), min_df=2), text_col),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler())]), numeric_cols)
    ],
    remainder="drop"
)

X_train_df, X_test_df, Y_train, Y_test = train_test_split(
    model_df[[text_col] + categorical_cols + numeric_cols],
    Y, test_size=0.20, random_state=SEED
)
X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)
if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
    X_test = X_test.toarray()
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")
print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)

## 11. Build the Artificial Neural Network (MLP)
Each output neuron corresponds to one role. Binary cross-entropy is applied independently across role labels. Dropout helps reduce overfitting.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.35),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(role_names), activation="sigmoid")
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="binary_accuracy")]
)
model.summary()

## 12. Train the model

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history = model.fit(
    X_train, Y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Training")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("Binary cross-entropy loss")
axes[0].legend()
axes[1].plot(history.history["binary_accuracy"], label="Training")
axes[1].plot(history.history["val_binary_accuracy"], label="Validation")
axes[1].set_title("Binary accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

## 13. Evaluate multi-label recommendations
A threshold of 0.5 is a baseline, not a universal optimum. In practice, tune it on a validation set and inspect precision/recall trade-offs. Subset accuracy is strict: every role label for a profile must match.

In [ ]:
y_prob = model.predict(X_test, verbose=0)
THRESHOLD = 0.50
y_pred = (y_prob >= THRESHOLD).astype(int)

metrics = pd.DataFrame({
    "metric": ["Micro precision","Micro recall","Micro F1","Macro precision",
               "Macro recall","Macro F1","Hamming loss","Subset accuracy"],
    "value": [
        precision_score(Y_test, y_pred, average="micro", zero_division=0),
        recall_score(Y_test, y_pred, average="micro", zero_division=0),
        f1_score(Y_test, y_pred, average="micro", zero_division=0),
        precision_score(Y_test, y_pred, average="macro", zero_division=0),
        recall_score(Y_test, y_pred, average="macro", zero_division=0),
        f1_score(Y_test, y_pred, average="macro", zero_division=0),
        hamming_loss(Y_test, y_pred),
        accuracy_score(Y_test, y_pred)
    ]
})
display(metrics)

In [ ]:
report = classification_report(Y_test, y_pred, target_names=role_names,
                               output_dict=True, zero_division=0)
per_role_metrics = pd.DataFrame(report).T.loc[role_names,
    ["precision","recall","f1-score","support"]].sort_values("f1-score", ascending=False)
display(per_role_metrics)

## 14. Recommendation and skill-gap functions

In [ ]:
role_skill_map = {row.job_role: set(row.core_skills.split("; "))
                  for row in occupations.itertuples(index=False)}

def recommend_roles(profile, model=model, preprocessor=preprocessor,
                    roles=role_names, threshold=0.35, top_k=5):
    \"\"\"profile: dict with resume_text, education_level, years_experience, domain_interest.\"\"\"
    one = pd.DataFrame([{
        "resume_text_clean": clean_text(profile.get("resume_text", "")),
        "education_level": profile.get("education_level", "Unknown"),
        "domain_interest": profile.get("domain_interest", "Unknown"),
        "years_experience": profile.get("years_experience", 0)
    }])
    features = preprocessor.transform(one)
    if hasattr(features, "toarray"):
        features = features.toarray()
    scores = model.predict(features.astype("float32"), verbose=0)[0]
    ranked = sorted(zip(roles, scores), key=lambda x: x[1], reverse=True)
    selected = [(r, float(s)) for r,s in ranked if s >= threshold][:top_k]
    if not selected:
        selected = [(r, float(s)) for r,s in ranked[:min(3, top_k)]]
    return pd.DataFrame(selected, columns=["recommended_role","recommendation_score"])

def analyze_skill_gap(candidate_skills, recommended_role):
    present = {clean_text(s) for s in candidate_skills}
    required = role_skill_map[recommended_role]
    matched = sorted(present & required)
    missing = sorted(required - present)
    coverage = len(matched) / max(len(required), 1)
    return {"role": recommended_role, "matched_skills": matched,
            "skills_to_develop": missing, "coverage": round(coverage, 3)}

## 15. Example candidate prediction

In [ ]:
sample_candidate = {
    "resume_text": "Computer engineering graduate. Python, SQL, Excel, Power BI, statistics, pandas and data visualization. Built dashboards and performed exploratory data analysis.",
    "education_level": "Bachelor's",
    "years_experience": 1,
    "domain_interest": "Technology"
}
sample_recommendations = recommend_roles(sample_candidate, threshold=0.30, top_k=5)
display(sample_recommendations)
sample_skills = ["python","sql","excel","power bi","statistics","pandas"]
if not sample_recommendations.empty:
    best_role = sample_recommendations.iloc[0]["recommended_role"]
    display(analyze_skill_gap(sample_skills, best_role))

## 16. Save trained artifacts
The notebook saves the Keras model and fitted preprocessing pipeline. Keep the role order file with the model because output neuron positions correspond to that exact order.

In [ ]:
os.makedirs("../models", exist_ok=True)
model.save("../models/job_role_ann.keras")
import joblib
joblib.dump(preprocessor, "../models/preprocessor.joblib")
with open("../models/role_names.json", "w", encoding="utf-8") as f:
    json.dump(role_names, f, indent=2)
print("Saved model, preprocessor and role order.")

## 17. Additional reusable prediction snippets

### Snippet A — Change candidate details
Edit the values and run `recommend_roles`.

In [ ]:
candidate_a = {
    "resume_text": "BSc graduate. Python, SQL, Tableau, Excel, statistics, data cleaning and dashboard development.",
    "education_level": "Bachelor's",
    "years_experience": 0,
    "domain_interest": "Finance"
}
recommend_roles(candidate_a, threshold=0.30, top_k=7)

### Snippet B — Paste resume text directly

In [ ]:
resume_text_input = \"\"\"
Paste the candidate's resume summary, skills, projects and experience here.
Example: Developed Python automation scripts, queried PostgreSQL databases,
created Power BI dashboards and performed statistical analysis.
\"\"\"
candidate_b = {
    "resume_text": resume_text_input,
    "education_level": "Bachelor's",
    "years_experience": 2,
    "domain_interest": "Technology"
}
recommend_roles(candidate_b, threshold=0.35, top_k=5)

### Snippet C — Use a custom list of skills and inspect gaps

In [ ]:
custom_skills = ["python", "sql", "excel", "power bi", "statistics"]
for role in ["Data Analyst", "Data Scientist", "Business Intelligence Developer"]:
    display(analyze_skill_gap(custom_skills, role))

### Snippet D — Adjust recommendation threshold

In [ ]:
# Lower threshold -> more recommendations; higher threshold -> fewer, stricter recommendations.
for cutoff in [0.20, 0.35, 0.50, 0.65]:
    print(f"Threshold = {cutoff}")
    display(recommend_roles(sample_candidate, threshold=cutoff, top_k=10))

### Snippet E — Batch-score several candidate profiles

In [ ]:
candidate_batch = [
    {"resume_text":"Java Spring SQL REST APIs Git unit testing", "education_level":"Bachelor's",
     "years_experience":2, "domain_interest":"Technology"},
    {"resume_text":"Linux networking SIEM incident response security monitoring Python",
     "education_level":"Bachelor's", "years_experience":1, "domain_interest":"Technology"}
]
for index, candidate in enumerate(candidate_batch, start=1):
    print(f"Candidate {index}")
    display(recommend_roles(candidate, threshold=0.30, top_k=5))

## 18. How the model works — from resume to recommendations

1. **Input:** resume text, skills, education, experience and domain interest.
2. **Text cleaning:** lowercases text, removes noisy punctuation and standardizes whitespace.
3. **TF-IDF:** turns words and two-word phrases into numeric features. Terms that are frequent in one resume but less common across all resumes receive more weight.
4. **Categorical encoding:** education and domain become indicator columns; experience is imputed and standardized.
5. **Dense neural layers:** each neuron computes a weighted sum plus bias, then ReLU introduces non-linearity. Batch normalization and dropout support stable training and regularization.
6. **Sigmoid output:** one output neuron per role converts its logit to a score between 0 and 1. These are model scores, not calibrated probabilities.
7. **Binary cross-entropy:** compares every role output with its 0/1 annotation and penalizes incorrect role predictions.
8. **Threshold/ranking:** scores are sorted. A threshold selects roles; top-k limits the number shown.
9. **Skill-gap analysis:** compares the candidate's stated skills with the educational role skill list and identifies skills to develop.

### Limitations and responsible use
- Synthetic labels are generated from role-skill overlap and a primary-role seed; this makes the task reproducible but can create artificial patterns and optimistic metrics.
- Scores are not evidence of employability, job availability, salary, seniority fit or hiring probability.
- The model may inherit bias from any future training data. Do not use it for automated hiring, rejection, ranking employees or other consequential decisions.
- Resume data is personal data. Obtain consent, minimize collection, protect storage and comply with applicable law.
- Validate on independent, human-reviewed, representative data before making claims about real-world performance.

**Educational career guidance only — recommendations are not guaranteed employment outcomes.**